# Data Mining Lab Assignment
## Apache Spark Data Lake Preprocessing

In [1]:
!pip install pyspark

In [2]:
import os
import shutil
import glob

from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder \
    .appName("DataMiningDataLake") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.4


## 1. Data Lake Structure

In [4]:
uploaded = files.upload()

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver", exist_ok=True)

shutil.copyfile(
    "sales.csv",
    "data_lake/bronze/sales.csv"
)

print("Bronze layer created successfully")

Saving sales.csv to sales.csv
Bronze layer created successfully


## 2. Data Loading and Inspection

In [5]:
bronze_path = "data_lake/bronze/sales.csv"

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(bronze_path)

print("Schema:")
df.printSchema()

print("First 5 rows:")
df.show(5, truncate=False)

print("Total records:", df.count())

Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

First 5 rows:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+---

## 3. Data Quality Assessment

In [6]:
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print("Total rows:", total_rows)
print("Duplicate rows:", total_rows - unique_rows)

Total rows: 1000
Duplicate rows: 10


In [7]:
df.select([
    F.count(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == ""),
            1
        )
    ).alias(c)
    for c in df.columns
]).show(truncate=False)

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [8]:
text_cols = [
    c for c, t in df.dtypes if t == "string"
]

for c in text_cols:
    print("Distinct values in", c)
    df.select(c).distinct().show(30, truncate=False)

Distinct values in customer_id
+-----------+
|customer_id|
+-----------+
|C0114      |
|C0141      |
|C0243      |
|C0034      |
|C0154      |
|C0347      |
|C0130      |
|C0338      |
|C0167      |
|C0179      |
|C0195      |
|C0150      |
|C0033      |
|C0107      |
|C0016      |
|C0131      |
|C0191      |
|C0261      |
|C0057      |
|C0280      |
|C0046      |
|C0284      |
|C0102      |
|C0061      |
|C0337      |
|C0028      |
|C0058      |
|C0050      |
|C0139      |
|C0109      |
+-----------+
only showing top 30 rows
Distinct values in customer_name
+------------------+
|customer_name     |
+------------------+
|Pranav Jain       |
|Tanvi Rao         |
|Meera Pandey      |
|Tanvi Patel       |
|Rahul Jain        |
|Kavya Bose        |
|Siddharth Menon   |
|Divya Naidu       |
|Aditi Agarwal     |
|Aman Naidu        |
|Siddharth Malhotra|
|Sneha Chopra      |
|Vikram Reddy      |
|Harsh Iyer        |
|Tanvi Naidu       |
|Swathi Gupta      |
|Shreya Gupta      |
|Harsh Pillai  

In [9]:
numeric_cols = [
    "quantity", "unit_price", "discount_percent"
]

for c in numeric_cols:
    df.select(
        F.min(c).alias("Minimum"),
        F.max(c).alias("Maximum"),
        F.avg(c).alias("Average")
    ).show()

+-------+-------+-------+
|Minimum|Maximum|Average|
+-------+-------+-------+
|     -2|     10|  5.355|
+-------+-------+-------+

+-------+-------+-------+
|Minimum|Maximum|Average|
+-------+-------+-------+
|   -500|  81010| 5702.4|
+-------+-------+-------+

+-------+-------+-------+
|Minimum|Maximum|Average|
+-------+-------+-------+
|     -5|    120| 19.948|
+-------+-------+-------+



In [10]:
spark.conf.set("spark.sql.ansi.enabled", "false")

df.select(
    F.to_date("order_date", "yyyy-MM-dd").alias("parsed_date")
).select(
    F.count(
        F.when(F.col("parsed_date").isNull(), 1)
    ).alias("Invalid Dates")
).show()

+-------------+
|Invalid Dates|
+-------------+
|            6|
+-------------+



## 4. Data Preprocessing

In [11]:
silver_df = df.dropDuplicates()

print("Rows before:", df.count())
print("Rows after:", silver_df.count())

Rows before: 1000
Rows after: 990


In [12]:
text_cols = [
    c for c, t in silver_df.dtypes if t == "string"
]

for c in text_cols:
    silver_df = silver_df.withColumn(
        c,
        F.when(
            F.trim(F.col(c)) == "",
            None
        ).otherwise(F.trim(F.col(c)))
    )

for c in text_cols:
    if c == "customer_id":
        silver_df = silver_df.withColumn(
            c, F.upper(F.col(c))
        )
    elif c != "order_date":
        silver_df = silver_df.withColumn(
            c, F.initcap(F.col(c))
        )

In [13]:
silver_df = silver_df.withColumn(
    "quantity",
    F.when(F.col("quantity") > 0, F.col("quantity"))
     .otherwise(None)
)

silver_df = silver_df.withColumn(
    "unit_price",
    F.when(F.col("unit_price") > 0, F.col("unit_price"))
     .otherwise(None)
)

silver_df = silver_df.withColumn(
    "discount_percent",
    F.when(
        F.col("discount_percent").between(0, 100),
        F.col("discount_percent")
    ).otherwise(None)
)

In [14]:
categorical_cols = [
    c for c in text_cols if c != "order_date"
]

numeric_cols = [
    "quantity", "unit_price", "discount_percent"
]

# Fill numerical missing values with median
for c in numeric_cols:
    median = silver_df.approxQuantile(c, [0.5], 0.01)

    if median and median[0] is not None:
        silver_df = silver_df.fillna({c: median[0]})

# Fill categorical missing values with mode
for c in categorical_cols:
    mode_rows = (
        silver_df.filter(F.col(c).isNotNull())
        .groupBy(c)
        .count()
        .orderBy(F.desc("count"), F.asc(c))
        .limit(1)
        .collect()
    )

    if mode_rows:
        mode_value = mode_rows[0][c]
        silver_df = silver_df.fillna({c: mode_value})
    else:
        silver_df = silver_df.fillna({c: "Unknown"})

In [15]:
silver_df = silver_df.withColumn(
    "order_date",
    F.date_format(
        F.to_date("order_date", "yyyy-MM-dd"),
        "yyyy-MM-dd"
    )
)

## 5. Verification

In [16]:
print("Rows before preprocessing:", df.count())
print("Rows after preprocessing:", silver_df.count())

print("Final Schema:")
silver_df.printSchema()

print("Cleaned Data:")
silver_df.show(10, truncate=False)

print("Remaining Null Values:")
silver_df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in silver_df.columns
]).show(truncate=False)

Rows before preprocessing: 1000
Rows after preprocessing: 990
Final Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = false)

Cleaned Data:
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order

## 6. Saving Data to Silver Layer

In [17]:
silver_path = "data_lake/silver/preprocessed_sales"

silver_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_path)

print("Silver data saved successfully")

Silver data saved successfully


In [18]:
silver_read = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(silver_path)

print("Silver Row Count:", silver_read.count())

silver_read.show(5, truncate=False)
silver_read.printSchema()

Silver Row Count: 990
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2       |670       |30              |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4       |670       |0               |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books    |6       |610       |12  

In [19]:
os.makedirs("submission/silver", exist_ok=True)

part_file = glob.glob(
    silver_path + "/part-*.csv"
)[0]

shutil.copyfile(
    part_file,
    "submission/silver/sales_silver.csv"
)

files.download("submission/silver/sales_silver.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>